In [1]:
!pip install river

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 6.0 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 4.6 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.1 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 7.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.6/12.3 MB 6.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/12.3 MB 6.0 MB/s eta 0:00:02
   ------------- -------------------------- 4.2/12.3 MB 5.2 MB/s eta 0:00:02
   ----------------- ---------------------- 5.2/12.3 MB 5.0 MB/s eta 0:00:02
   -------------------- ------------------- 6.3/12.3 MB 5.0 MB/s eta 0:00:02
   ---------------------- ----------------- 6.8/12.3 MB 4.7 MB/s eta 0:00:02
   ------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
from datetime import datetime, timedelta
from river.anomaly import HalfSpaceTrees
from river import preprocessing

ANOMALY_THRESHOLD = 0.7
N_TREES = 25
HEIGHT = 8
WINDOW_SIZE = 100

scaler = preprocessing.StandardScaler()
model = HalfSpaceTrees(n_trees=N_TREES, height=HEIGHT, window_size=WINDOW_SIZE, seed=42)

In [3]:
def extract_features(log_entry):
    return {
        "failed_attempts_last_60s":   log_entry["failed_attempts_last_60s"],
        "unique_ips_last_60s":        log_entry["unique_ips_last_60s"],
        "time_since_last_attempt_s":  log_entry["time_since_last_attempt_s"],
        "is_success":                 int(log_entry["success"]),
        "fail_rate":                  log_entry["failed_attempts_last_60s"] /
                                      max(log_entry["unique_ips_last_60s"], 1),
    }

In [4]:
def process_login_event(log_entry):
    features = extract_features(log_entry)
    scaler.learn_one(features)
    scaled = scaler.transform_one(features)
    score = model.score_one(scaled)   # score BEFORE learn
    model.learn_one(scaled)
    return {
        "user_id":        log_entry.get("user_id", "unknown"),
        "ip_address":     log_entry.get("ip_address", "unknown"),
        "timestamp":      log_entry.get("timestamp", datetime.now().isoformat()),
        "anomaly_score":  round(score, 4),
        "is_brute_force": score >= ANOMALY_THRESHOLD,
    }

def is_brute_force(log_entry):
    return process_login_event(log_entry)["is_brute_force"]

In [5]:
# TODO: Replace with real MongoDB log preprocessing
def generate_synthetic_logs(n_normal=300, n_attack=50):
    logs = []
    base_time = datetime.now()

    for i in range(n_normal):
        logs.append({
            "user_id":                   f"user_{random.randint(1, 50)}",
            "ip_address":                f"192.168.1.{random.randint(1, 255)}",
            "timestamp":                 (base_time + timedelta(seconds=i*10)).isoformat(),
            "success":                   random.random() > 0.1,
            "failed_attempts_last_60s":  random.randint(0, 2),
            "unique_ips_last_60s":       random.randint(1, 3),
            "time_since_last_attempt_s": random.uniform(30, 300),
            "label": "normal"
        })

    for i in range(n_attack):
        logs.append({
            "user_id":                   f"user_{random.randint(1, 5)}",
            "ip_address":                f"10.0.0.{random.randint(1, 10)}",
            "timestamp":                 (base_time + timedelta(seconds=i*2)).isoformat(),
            "success":                   random.random() > 0.95,
            "failed_attempts_last_60s":  random.randint(15, 40),
            "unique_ips_last_60s":       random.randint(1, 3),
            "time_since_last_attempt_s": random.uniform(0.5, 3),
            "label": "brute_force"
        })

    random.shuffle(logs)
    return logs

logs = generate_synthetic_logs()
print(f"Loaded {len(logs)} logs")

Loaded 350 logs


In [6]:
tp = fp = tn = fn = 0

for entry in logs:
    result = process_login_event(entry)
    truth = entry["label"]
    if result["is_brute_force"] and truth == "brute_force": tp += 1
    elif result["is_brute_force"] and truth == "normal":     fp += 1
    elif not result["is_brute_force"] and truth == "normal": tn += 1
    else:                                                     fn += 1

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}  FP: {fp}  TN: {tn}  FN: {fn}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

TP: 38  FP: 198  TN: 102  FN: 12
Precision : 0.1610
Recall    : 0.7600
F1 Score  : 0.2657
